# MM4250: take data

One **take** = the switch's internal standards (ALL_OPEN, short, load), each chosen channel, then the standards again (the drift check), `repeats` times (from the cooldown file). Every position is saved raw (all four S-parameters) to the lab database (`<project>_Cooldown_<NN>_vna.db`), then each channel is corrected with NIST's definitions (295 K or 3 K, picked from the fridge temperature) and saved as its own runs, linked to the raw ones.

- Run the cells top to bottom. Everything about the setup (ports, fridge lines, sweep) is in the cooldown file; this notebook only says which file.
- Only one notebook can hold the switch: run the other notebook's **Close** cell first.
- VNA port 1 on the input line, port 2 on the return line (see the file's `lines`). The LNA must be on, or nothing comes back.
- The switch is always left `ALL_OPEN` at the end of a take, even if it fails. See `README.md`.

**Setup**: which cooldown file. Prints what it says; fix the file if anything is wrong.

In [ ]:
COOLDOWN_FILE = "cooldowns/CD05.json"   # this cooldown's file (see README.md)

import sys
from pathlib import Path
try:
    here = Path(_dh[0])                      # this notebook's folder (Jupyter)
except NameError:
    here = Path.cwd()
root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent                       # the folder with drivers/ in it
sys.path[:0] = [str(here), str(root)]
%matplotlib inline

import vna, db, ecal, plots
cfg = db.load_cooldown(here / COOLDOWN_FILE)
db.print_cooldown(cfg)

**Connect**: VNA, switch, fridge (optional: without the VPN it warns and carries on, with no temperatures), and the database.

In [ ]:
from qcodes.instrument import Instrument
from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250

Instrument.close_all()                       # so re-running this cell is safe
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")                    # connecting sets ALL_OPEN

fridge = None                                # only for temperatures: optional
try:
    from maybell_qcodes import MaybellFridge
    fridge = MaybellFridge("fridge", host="10.179.200.168")   # needs the GlobalConnect VPN
except Exception as e:
    print(f"[warning] no fridge ({type(e).__name__}: {e}); carrying on without temperatures")
print("Fridge (mixing plate):", vna.temp_label(vna.read_temperature(fridge)))

DB = db.open_database(cfg, "vna")            # E:\\QCoDeS\\<project>\\Cooldown_<NN>\\..._vna on the DAQ

**Take**: about a minute per channel. `CHANNELS = None` measures every port in the file; `[4]` just RF4 (still with both sets of standards). Prints the drift at the end: compare with NIST's few 1e-2.

In [ ]:
CHANNELS = None          # None = every port in the cooldown file; or e.g. [4]
NOTE = ""                # anything about this take: fridge state, why you took it
SAVE_TOUCHSTONE = False  # True: also .s2p/.s1p files in Sweeps/takes/ beside the database
DEFINITIONS = None       # None = NIST 295K or 3K from the fridge temperature; or "295K" / "3K"

save = db.Saver(DB, "take", files=SAVE_TOUCHSTONE)
t = vna.take(cfg, ksvna, switch, fridge, channels=CHANNELS, note=NOTE, save=save)   # ends ALL_OPEN
take = db.load_take(t["take_id"], DB)
result = ecal.correct_take(take, DEFINITIONS)          # None: no temperature and no DEFINITIONS
if result is not None:
    db.save_corrected(DB, result, take, files=SAVE_TOUCHSTONE)
    take = db.load_take(t["take_id"], DB)
db.backup_db(DB)
if SAVE_TOUCHSTONE:
    print("Files in", db.take_folder(DB, take))
ecal.drift(take);

**Plot latest take**: run right after Take. Left: raw trace the correction uses; right: corrected at each connector. |S| in dB on top, phase below.

In [ ]:
SAVE_FIGURE = False      # True: take.png in the take's Sweeps/takes/ folder (never overwrites)

plots.plot_take(take, save=db.take_folder(DB, take) / "take.png" if SAVE_FIGURE else None);

**Recall**: any time, no instruments needed. Lists every take in the database, then plots one (or a list of runs) and can export Touchstone files.

In [ ]:
RECALL_DB = DB           # or another database, e.g. an earlier cooldown's path
TAKE_ID = None           # None = the newest take; or one from the list, e.g. "20261108T141502.318"
RUN_IDS = None           # or a list of run ids instead of a take, e.g. [12, 13]
EXPORT = False           # True: Touchstone files in Sweeps/export/ beside that database (never overwrites)
CORRECT_WITH = None      # "295K" or "3K": correct a take that was saved raw only (no temperature then)

db.list_takes(RECALL_DB)
if RUN_IDS:
    plots.plot_runs(db.load_runs(RUN_IDS, RECALL_DB))
    ids = RUN_IDS
else:
    old = db.load_take(TAKE_ID, RECALL_DB)
    if CORRECT_WITH:
        db.save_corrected(RECALL_DB, ecal.correct_take(old, CORRECT_WITH), old)
        old = db.load_take(old["take_id"], RECALL_DB)
    plots.plot_take(old)
    ids = db.take_run_ids(old)
if EXPORT:
    db.export_touchstone(ids, RECALL_DB)

**Close**: before anyone else needs the switch or VNA. Backs up the database (skipped if nothing is new), leaves the switch `ALL_OPEN` and releases everything.

In [ ]:
db.backup_db(DB)
vna.leave_switch_open(ksvna, switch)          # source off for the move
print("switch, last sent:", switch.state.cache.get(get_if_invalid=False))   # not read back: the lab driver's readback lags
ksvna.output(False)
ksvna.close()
switch.close()
if fridge is not None:
    fridge.close()